# Guided Project: PDF-Based Knowledge Base RAG System
## Participant Notebook

### Objective
Build a Retrieval-Augmented Generation (RAG) application that answers questions about a PDF document using **only** the content of that document.

### Required pipeline
**PDF -> PyPDFLoader -> Chunking -> Embeddings -> ChromaDB -> Retrieval -> GPT-4o-mini -> Answer**

### How to work through this notebook
1. Run **Task 0** to confirm the environment is ready.
2. For each task: read the goal and hints, then complete the `# TODO` cells and run them.
3. Check your output against **Expected result**.
4. Tick the boxes in the **Final Validation Checklist** at the end.

### Ground rules
- The VM is preconfigured. Do **not** run `pip install`.
- The OpenAI API key is configured by the lab administrator.
- Use only the supplied PDF: `../04_DATA/sample_knowledge_base.pdf`.

In [ ]:
# Task 0 - Environment preflight (run this first)
# The VM is preconfigured. This cell only VERIFIES that everything is in place.
import importlib, os, sys

REQUIRED = [
    "langchain", "langchain_openai", "langchain_community",
    "langchain_text_splitters", "langchain_chroma", "chromadb",
    "pypdf", "dotenv",
]
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]

from dotenv import load_dotenv
load_dotenv("../05_CONFIG/.env")   # vendor-managed key; safe no-op if absent

key = os.getenv("OPENAI_API_KEY", "")
data_ok = os.path.isfile("../04_DATA/sample_knowledge_base.pdf")

print("packages   :", "OK" if not missing else f"MISSING -> {missing}")
print("OPENAI key  :", "OK" if key.startswith("sk-") else "NOT CONFIGURED - contact the lab administrator")
print("sample PDF  :", "OK" if data_ok else "MISSING at ../04_DATA/sample_knowledge_base.pdf")

assert not missing, f"Preconfigured VM is missing packages: {missing}"
assert data_ok, "Supplied PDF not found."
print("\nPreflight passed - continue with Task 1.")

# Project Architecture

```text
PDF file
  -> PyPDFLoader                     (load pages + metadata)
  -> RecursiveCharacterTextSplitter  (chunk_size=1000, chunk_overlap=200)
  -> OpenAIEmbeddings                (text-embedding-3-small)
  -> Chroma                          (persistent vector store at ../chroma_db)
  -> retriever                       (similarity search, k=3)
  -> ChatPromptTemplate             (context + question, grounded)
  -> ChatOpenAI                      (gpt-4o-mini, temperature=0)
  -> Answer + source pages
```

A larger annotated diagram is in `../06_REFERENCE/architecture.md`.

## Task 1 - Load the PDF

### Goal
Load the supplied PDF using LangChain's PDF loader.

### Your tasks
1. Import the PDF loader.
2. Set the PDF path.
3. Create the loader.
4. Load the documents.
5. Print the number of pages, the first page's text, and its metadata.

### Hints
- The loader is in `langchain_community.document_loaders`.
- Its class name starts with `PyPDF...`.
- It exposes a `.load()` method that returns a list of `Document` objects.

### Expected result
The page count prints (the sample PDF has a few pages), followed by readable text from page 1 and a metadata dict containing `source` and `page`.

In [ ]:
# Task 1 - Import the PDF loader
# TODO: complete the import


In [ ]:
# Task 1 - Set the PDF path
pdf_path = "../04_DATA/sample_knowledge_base.pdf"


In [ ]:
# Task 1 - Create the loader
# TODO: create the loader using pdf_path


In [ ]:
# Task 1 - Load the documents
# TODO: call .load() and keep the result in `documents`


In [ ]:
# Task 1 - Inspect the loaded document
# TODO: print len(documents), documents[0].page_content, documents[0].metadata


## Task 2 - Split the Document into Chunks

### Goal
Split the loaded PDF into fixed-size, overlapping chunks.

### Required configuration
- Chunk size: **1000**
- Chunk overlap: **200**

### Your tasks
1. Import `RecursiveCharacterTextSplitter` from `langchain_text_splitters`.
2. Create the splitter with the required configuration.
3. Split `documents` into `chunks`.
4. Print the number of chunks and the first 2 chunks (content + metadata).

### Mini experiment (after it works)
Try `chunk_size=500, chunk_overlap=100` and compare the chunk count. Restore 1000/200 before continuing.

In [ ]:
# Task 2 - Import the text splitter
# TODO


In [ ]:
# Task 2 - Configure chunking
chunk_size = 1000
chunk_overlap = 200


In [ ]:
# Task 2 - Create the splitter
# TODO


In [ ]:
# Task 2 - Split the documents into `chunks`
# TODO


In [ ]:
# Task 2 - Inspect the chunks
# TODO: print len(chunks) and chunks[:2]


## Task 3 - Generate Embeddings and Store in ChromaDB

### Goal
Embed the chunks with `text-embedding-3-small` and store them in a persistent Chroma vector store.

### Your tasks
1. Import `OpenAIEmbeddings` (from `langchain_openai`) and `Chroma` (from `langchain_chroma`).
2. Create the embedding model using `text-embedding-3-small`.
3. Create a Chroma store from `chunks`, persisting to `../chroma_db` (use `collection_name="pdf_kb"`).
4. Run a similarity search for a test query and print the top 3 chunks.

### Hints
- `Chroma.from_documents(documents=..., embedding=..., persist_directory=..., collection_name=...)`
- `vectorstore.similarity_search(query, k=3)`
- Re-running `from_documents` adds the chunks **again**. If you re-run this task, either delete `../chroma_db` first or load the existing store with `Chroma(persist_directory=..., embedding_function=..., collection_name=...)`.

In [ ]:
# Task 3 - Imports
# TODO


In [ ]:
# Task 3 - Create the embedding model
# TODO: model = text-embedding-3-small


In [ ]:
# Task 3 - Create the persistent vector store
# TODO: persist to "../chroma_db", collection_name="pdf_kb"


In [ ]:
# Task 3 - Test similarity search
query = "How many casual leave days are provided each year?"
# TODO: run similarity_search(query, k=3) and print the retrieved chunks


## Task 4 - Build the RAG Question-Answering Pipeline

### Goal
Connect retrieval to GPT-4o-mini to produce **grounded** answers.

### Required flow
Question -> Retriever (k=3) -> Context -> Grounded prompt -> GPT-4o-mini -> Answer + sources

### Your tasks
1. Import `ChatOpenAI` and `ChatPromptTemplate`.
2. Build a retriever from your vector store with `k=3`.
3. Initialise `ChatOpenAI(model="gpt-4o-mini", temperature=0)`.
4. Write a prompt that forces the model to answer **only** from the context.
5. Ask a question, retrieve chunks, build the context string, invoke the LLM.
6. Print the answer and the **de-duplicated** source page numbers.

### Grounding requirement
If the answer is not in the retrieved context, the model must reply exactly:

> I could not find the answer in the provided document.

In [ ]:
# Task 4 - Imports
# TODO: ChatOpenAI, ChatPromptTemplate


In [ ]:
# Task 4 - Retriever (k=3)
# TODO


In [ ]:
# Task 4 - Initialise GPT-4o-mini (temperature=0)
# TODO


In [ ]:
# Task 4 - Grounded prompt
# TODO: ChatPromptTemplate.from_template(...) with {context} and {question}


In [ ]:
# Task 4 - Ask, retrieve, build context, invoke
question = "What is the standard notice period for regular full-time employees?"
# TODO


In [ ]:
# Task 4 - Print answer + de-duplicated source pages
# TODO


## Task 5 - Build an Interactive RAG Application

### Goal
Combine Tasks 1-4 into a single script, `rag_app.py`, that answers questions interactively from the terminal.

A starter file is provided at `../02_STARTER_CODE/rag_app.py`. Complete the `# TODO` sections, then run it from a terminal:

```bash
cd 02_STARTER_CODE
python rag_app.py
```

### The app must
1. Load the PDF, chunk it, embed it, store it in Chroma (load the store if it already exists).
2. Create a `k=3` retriever and the grounded GPT-4o-mini chain.
3. Loop: read a question, print a grounded answer + source pages.
4. Exit when the user types `exit`.

### Validate with `../04_DATA/RAG_Question_Set.pdf`
- A document-grounded question (e.g. Q2) -> correct answer + pages.
- A retrieval/reasoning question (e.g. Q11) -> correct answer.
- A negative question (e.g. Q17) -> the refusal sentence, no invented answer.

In [ ]:
# Task 5 - Optional scratch space
# Develop rag_app.py in ../02_STARTER_CODE/rag_app.py, not here.
# You can prototype snippets in this cell if useful.


## Final Validation Checklist

- [ ] Task 0 preflight passed.
- [ ] PDF loads; page count printed.
- [ ] Chunks created with size **1000** and overlap **200**.
- [ ] `text-embedding-3-small` used for embeddings.
- [ ] Chroma store persisted at `../chroma_db` (collection `pdf_kb`).
- [ ] Similarity search returns relevant chunks.
- [ ] `gpt-4o-mini` with `temperature=0` used for answers.
- [ ] Answers use **only** retrieved context.
- [ ] Source page numbers displayed (de-duplicated).
- [ ] Negative / out-of-context question returns the refusal sentence.
- [ ] `rag_app.py` runs and answers interactively.

When every box is ticked, run `python ../scripts/validate_solution.py` for an automated check.